In [0]:
query = (autoloader_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("ledgr.bronze.sessions_raw_autoloader")
)

query.awaitTermination()

new_count = spark.table("ledgr.bronze.sessions_raw_autoloader").count()
print("Row count after adding 10th file: " + str(new_count))
print("Original count: 10056")
print("Expected new count (10056 + rows in duplicated file): should increase by exactly the duplicate file's row count, NOT double everything")

In [0]:
# Optional: drop the test table entirely since it was just for proving the concept,
# real Bronze work continues to use ledgr.bronze.sessions_raw
spark.sql("DROP TABLE IF EXISTS ledgr.bronze.sessions_raw_autoloader")
print("Dropped test Auto Loader table")

In [0]:
dbt_result = spark.sql("SELECT * FROM ledgr.gold.mart_cost_per_success ORDER BY total_cost_usd DESC LIMIT 5")
dbt_result.show(truncate=False)

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("ledgr.silver.calls_enriched")
date_range = silver_df.select(
    F.min(F.to_date("start_time")).alias("earliest"),
    F.max(F.to_date("start_time")).alias("latest"),
    F.countDistinct(F.to_date("start_time")).alias("distinct_days")
).collect()[0]

print("Earliest: " + str(date_range.earliest))
print("Latest: " + str(date_range.latest))
print("Distinct days: " + str(date_range.distinct_days))

In [0]:
anomalies = spark.sql("""
    SELECT model_request, call_date, cost_per_success, rolling_7day_mean, 
           rolling_7day_stddev, is_anomaly
    FROM ledgr.gold.mart_cost_anomalies
    WHERE is_anomaly = true
    ORDER BY model_request, call_date
""")
anomalies.show(20, truncate=False)

total_rows = spark.sql("SELECT COUNT(*) as c FROM ledgr.gold.mart_cost_anomalies").collect()[0].c
anomaly_count = spark.sql("SELECT COUNT(*) as c FROM ledgr.gold.mart_cost_anomalies WHERE is_anomaly = true").collect()[0].c
print("Total rows: " + str(total_rows))
print("Anomalies flagged: " + str(anomaly_count))

In [0]:
from ledgr_databricks.silver_transform import (
    explode_bronze_sessions, extract_call_fields, validate_config_coverage,
    compute_injection_probability
)

bronze_df = spark.table("ledgr.bronze.sessions_raw")
exploded_df = explode_bronze_sessions(bronze_df)
normalized_df = extract_call_fields(exploded_df)
validate_config_coverage(normalized_df)
injected_df = compute_injection_probability(normalized_df)

audit_df = injected_df.select(
    "call_id", "attempt_id", "harness", "model_request",
    "harness_rate", "model_rate", "relative_risk", 
    "injection_probability", "hash_uniform", "is_selected_for_injection"
)

audit_df.write.format("delta").mode("overwrite").saveAsTable("ledgr.silver.injection_calibration_audit")
print("Audit table created: " + str(audit_df.count()) + " rows")

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("ledgr.silver.calls_enriched")

# Check: does session-level 'success' correlate with call-level 'outcome_state'?
reconciliation = (silver_df
    .groupBy("task_id", "success")
    .agg(
        F.count("*").alias("total_calls"),
        F.sum(F.when(F.col("outcome_state") == "SUCCESS", 1).otherwise(0)).alias("successful_calls")
    )
    .withColumn("all_calls_match_session_success",
        F.when(F.col("success") == True, F.col("successful_calls") == F.col("total_calls"))
         .otherwise(F.col("successful_calls") == 0)
    )
)

mismatch_count = reconciliation.filter(F.col("all_calls_match_session_success") == False).count()
total_sessions = reconciliation.count()

print("Total sessions checked: " + str(total_sessions))
print("Sessions where success/outcome_state DON'T fully align: " + str(mismatch_count))
print("Mismatch rate: " + str(round(mismatch_count / total_sessions * 100, 2)) + "%")

In [0]:
result = spark.sql("SELECT * FROM ledgr.gold.mart_success_outcome_reconciliation").collect()[0]
print("Total sessions: " + str(result.total_sessions))
print("Mismatched sessions: " + str(result.mismatched_sessions))
print("Mismatch rate: " + str(result.mismatch_rate_pct) + "%")
print("Expected: 10056 total, 6342 mismatched, 63.07%")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Row

silver_df = spark.table("ledgr.silver.calls_enriched")
existing_sample = silver_df.limit(5).collect()

before_count = spark.table("ledgr.silver.calls_enriched").count()

# Build 3 genuinely new fake rows (new attempt_ids/call_ids that don't exist yet),
# based on an existing row's structure so the schema matches
template_row = existing_sample[0].asDict()

new_rows_data = []
for i in range(1, 4):
    row_copy = dict(template_row)
    row_copy["attempt_id"] = "fake_new_attempt_" + str(i)
    row_copy["call_id"] = "fake_call_" + str(i)
    new_rows_data.append(Row(**row_copy))

new_rows_df = spark.createDataFrame(new_rows_data, schema=silver_df.schema)

# Mixed batch: 5 EXISTING rows (should NOT duplicate) + 3 genuinely NEW rows (SHOULD insert)
existing_rows_df = spark.createDataFrame(existing_sample, schema=silver_df.schema)
mixed_batch = existing_rows_df.unionByName(new_rows_df)

delta_table = DeltaTable.forName(spark, "ledgr.silver.calls_enriched")
(delta_table.alias("target")
    .merge(mixed_batch.alias("source"), "target.attempt_id = source.attempt_id")
    .whenNotMatchedInsertAll()
    .execute())

after_count = spark.table("ledgr.silver.calls_enriched").count()

print("Before: " + str(before_count))
print("After: " + str(after_count))
print("Increase: " + str(after_count - before_count))
print("Expected: exactly 3 (only new rows inserted, 5 existing rows correctly skipped)")

In [0]:
spark.sql("""
    DELETE FROM ledgr.silver.calls_enriched 
    WHERE attempt_id IN ('fake_new_attempt_1', 'fake_new_attempt_2', 'fake_new_attempt_3')
""")

final_count = spark.table("ledgr.silver.calls_enriched").count()
print("Row count after cleanup: " + str(final_count))
print("Expected: 265311 (back to original)")

In [0]:
result = spark.sql("SELECT * FROM ledgr.gold.mart_success_outcome_reconciliation").collect()[0]
print("Total sessions: " + str(result.total_sessions))
print("Mismatched sessions: " + str(result.mismatched_sessions))
print("Mismatch rate: " + str(result.mismatch_rate_pct) + "%")